In [1]:
import triton.runtime.build as tb
import subprocess

_orig_build = tb._build

def _loud_build(name, src, srcdir, library_dirs, include_dirs, libraries, ccflags):
    import os
    cc = os.environ.get("CC", "gcc")
    suffix = ".so"
    so = os.path.join(srcdir, f"{name}.so")

    cc_cmd = [cc, src, "-O3", "-shared", "-fPIC", "-Wno-psabi", "-o", so]
    cc_cmd += [f"-L{d}" for d in library_dirs]
    cc_cmd += [f"-l:libcuda.so.1"]
    cc_cmd += [f"-I{d}" for d in include_dirs if d]

    result = subprocess.run(cc_cmd, capture_output=True, text=True)
    print("STDOUT:", result.stdout)
    print("STDERR:", result.stderr)
    print("Return code:", result.returncode)
    return _orig_build(name, src, srcdir, library_dirs, include_dirs, libraries, ccflags)

tb._build = _loud_build

In [2]:
import torch
import triton
import triton.language as tl

@triton.jit
def my_kernel(x_ptr, y_ptr, N, BLOCK: tl.constexpr):
    pid = tl.program_id(0)
    offsets = pid * BLOCK + tl.arange(0, BLOCK)
    mask = offsets < N
    x = tl.load(x_ptr + offsets, mask=mask)
    tl.store(y_ptr + offsets, x * 2.0, mask=mask)

x = torch.randn(1024 * 1024, device='cuda')
y = torch.empty_like(x)
grid = (triton.cdiv(1024 * 1024, 1024),)

# Warmup (compila y cachea)
for _ in range(10):
    my_kernel[grid](x, y, 1024 * 1024, BLOCK=1024)

# Benchmark real
start = torch.cuda.Event(enable_timing=True)
end = torch.cuda.Event(enable_timing=True)
start.record()
for _ in range(100):
    my_kernel[grid](x, y, 1024 * 1024, BLOCK=1024)
end.record()
torch.cuda.synchronize()
print(f"Latency: {start.elapsed_time(end) / 100:.2f} ms")

STDOUT: 
STDERR: /tmp/tmp8k2jyndl/cuda_utils.c:7:10: fatal error: Python.h: No such file or directory
    7 | #include <Python.h>
      |          ^~~~~~~~~~
compilation terminated.

Return code: 1
STDOUT: 
STDERR: /tmp/tmpj8umwhfc/__triton_launcher.c:7:10: fatal error: Python.h: No such file or directory
    7 | #include <Python.h>
      |          ^~~~~~~~~~
compilation terminated.

Return code: 1
Latency: 0.03 ms
